#### Librerias

In [19]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
from scipy import sparse
import joblib
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from scipy.stats import uniform

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### Variables

In [20]:
MODELOS_PATH = "/content/drive/MyDrive/Trabajo práctico 3/modelos"

#### Carga de datos

Esta notebook trabaja solo con train y validación. El test manual (359 tweets)
no se usa acá — queda reservado para 06_evaluacion_final, donde se compara
contra TextBlob y se analiza similitud coseno.

In [21]:
X_train_bow = sparse.load_npz(f"{MODELOS_PATH}/X_train_bow.npz")
X_val_bow = sparse.load_npz(f"{MODELOS_PATH}/X_val_bow.npz")

X_train_tfidf = sparse.load_npz(f"{MODELOS_PATH}/X_train_tfidf.npz")
X_val_tfidf = sparse.load_npz(f"{MODELOS_PATH}/X_val_tfidf.npz")

y_train = np.load(f"{MODELOS_PATH}/y_train.npy")
y_val = np.load(f"{MODELOS_PATH}/y_val.npy")

X_train_bow.shape, X_val_bow.shape

((1360000, 20000), (240000, 20000))

#### Preparar la búsqueda con train/validación reales

PredefinedSplit le dice a la búsqueda exactamente qué filas son de entrenamiento
y cuáles de validación — entrena siempre con los 1.36M de train, mide siempre
contra los 240k de validación, una sola vez por combinación probada. Nada de
muestreo ni de repetir la búsqueda varias veces (cross-validation), porque ya
tenemos una validación real y grande.

In [22]:
X_busqueda_bow = sparse.vstack([X_train_bow, X_val_bow]).tocsr()
X_busqueda_tfidf = sparse.vstack([X_train_tfidf, X_val_tfidf]).tocsr()
y_busqueda = np.concatenate([y_train, y_val])

test_fold = np.concatenate([-np.ones(len(y_train)), np.zeros(len(y_val))])
ps = PredefinedSplit(test_fold)

#### Optimización — Naive Bayes

In [23]:
param_dist_nb = {"alpha": uniform(loc=0.01, scale=2.0)}

random_search_nb = RandomizedSearchCV(
    MultinomialNB(),
    param_distributions=param_dist_nb,
    n_iter=8,
    cv=ps,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
    refit=False
)

random_search_nb.fit(X_busqueda_bow, y_busqueda)
random_search_nb.best_params_, random_search_nb.best_score_

({'alpha': np.float64(1.9114286128198323)}, np.float64(0.7701458333333333))

`best_score_` es el accuracy sobre los 240k de validación real con el mejor alpha
encontrado. `refit=False` porque el reentrenamiento final lo hacemos a mano en la
celda siguiente, usando SOLO train — así la validación queda intacta y disponible
para medir después, en vez de gastarse dentro de la búsqueda.

#### Modelo final de Naive Bayes — entrenado con train, medido contra validación

In [24]:
mejor_alpha = random_search_nb.best_params_["alpha"]

modelo_nb_opt = MultinomialNB(alpha=mejor_alpha)
modelo_nb_opt.fit(X_train_bow, y_train)

pred_nb_opt_val = modelo_nb_opt.predict(X_val_bow)
print(classification_report(y_val, pred_nb_opt_val, target_names=["Negativo", "Positivo"]))

              precision    recall  f1-score   support

    Negativo       0.77      0.78      0.77    120000
    Positivo       0.77      0.76      0.77    120000

    accuracy                           0.77    240000
   macro avg       0.77      0.77      0.77    240000
weighted avg       0.77      0.77      0.77    240000



El modelo se entrena solo con train (1.36M) — la validación se deja intacta a
propósito para medir contra datos que el modelo genuinamente nunca vio. Este es
el número confiable de performance del proyecto para Naive Bayes.

#### Optimización — Logistic Regression

In [25]:
param_dist_lr = {"C": uniform(loc=0.01, scale=10)}

random_search_lr = RandomizedSearchCV(
    LogisticRegression(max_iter=1000),
    param_distributions=param_dist_lr,
    n_iter=8,
    cv=ps,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
    refit=False
)

random_search_lr.fit(X_busqueda_tfidf, y_busqueda)
random_search_lr.best_params_, random_search_lr.best_score_

({'C': np.float64(1.5699452033620265)}, np.float64(0.78285))

#### Modelo final de Logistic Regression — entrenado con train, medido contra validación

In [26]:
mejor_c = random_search_lr.best_params_["C"]

modelo_lr_opt = LogisticRegression(C=mejor_c, max_iter=1000)
modelo_lr_opt.fit(X_train_tfidf, y_train)

pred_lr_opt_val = modelo_lr_opt.predict(X_val_tfidf)
print(classification_report(y_val, pred_lr_opt_val, target_names=["Negativo", "Positivo"]))

              precision    recall  f1-score   support

    Negativo       0.79      0.76      0.78    120000
    Positivo       0.77      0.80      0.79    120000

    accuracy                           0.78    240000
   macro avg       0.78      0.78      0.78    240000
weighted avg       0.78      0.78      0.78    240000



#### Comparación: baseline vs optimizado (medido sobre validación, la vara confiable)

In [27]:
modelo_nb_base = joblib.load(f"{MODELOS_PATH}/modelo_nb_baseline.pkl")
modelo_lr_base = joblib.load(f"{MODELOS_PATH}/modelo_lr_baseline.pkl")

pred_nb_base_val = modelo_nb_base.predict(X_val_bow)
pred_lr_base_val = modelo_lr_base.predict(X_val_tfidf)

print("=== NAIVE BAYES — Validación ===")
print("--- Baseline ---")
print(classification_report(y_val, pred_nb_base_val, target_names=["Negativo", "Positivo"]))
print("--- Optimizado ---")
print(classification_report(y_val, pred_nb_opt_val, target_names=["Negativo", "Positivo"]))

print("\n=== LOGISTIC REGRESSION — Validación ===")
print("--- Baseline ---")
print(classification_report(y_val, pred_lr_base_val, target_names=["Negativo", "Positivo"]))
print("--- Optimizado ---")
print(classification_report(y_val, pred_lr_opt_val, target_names=["Negativo", "Positivo"]))

=== NAIVE BAYES — Validación ===
--- Baseline ---
              precision    recall  f1-score   support

    Negativo       0.77      0.78      0.77    120000
    Positivo       0.77      0.76      0.77    120000

    accuracy                           0.77    240000
   macro avg       0.77      0.77      0.77    240000
weighted avg       0.77      0.77      0.77    240000

--- Optimizado ---
              precision    recall  f1-score   support

    Negativo       0.77      0.78      0.77    120000
    Positivo       0.77      0.76      0.77    120000

    accuracy                           0.77    240000
   macro avg       0.77      0.77      0.77    240000
weighted avg       0.77      0.77      0.77    240000


=== LOGISTIC REGRESSION — Validación ===
--- Baseline ---
              precision    recall  f1-score   support

    Negativo       0.79      0.76      0.78    120000
    Positivo       0.77      0.80      0.79    120000

    accuracy                           0.78    240000


Comparación directa, mismo conjunto de validación (240k) para baseline y
optimizado en los dos modelos — misma vara de medir. Si la diferencia es chica,
confirma que el ajuste de hiperparámetros no tenía mucho margen para mejorar
sobre un dataset ya tan grande.

#### Guardado de modelos optimizados

In [28]:
joblib.dump(modelo_nb_opt, f"{MODELOS_PATH}/modelo_nb_optimizado.pkl")
joblib.dump(modelo_lr_opt, f"{MODELOS_PATH}/modelo_lr_optimizado.pkl")

['/content/drive/MyDrive/Trabajo práctico 3/modelos/modelo_lr_optimizado.pkl']